# 2025-08-02: Process final BMMC objects (whole and per-lineage)
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology
**Main aim**: Runs the standard scanpy processing pipeline on the final labelled BMMC object three ways: without Harmony, with Harmony batch correction, and per-L1 lineage (both with and without Harmony). Produces analysis-ready objects.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce

sc.settings.n_jobs = 30
sc.settings.verbosity = 0

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import sys
sys.path.insert(0, '../../00-utilities/functions/python/')
from process_scrna_data import process_adata

## 2. Process object without Harmony

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
adata = process_adata(adata, resolution=0.5, run_rank_genes=True)
adata.write('../../../data/rna/final-objects/final-bmmc-processed.h5ad')

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_tsne.py:126: UserWarning: In previous versions of scanpy, calling tsne with n_jobs > 1 would use MulticoreTSNE. Now this uses the scikit-learn version of TSNE by default. If you'd like the old behaviour (which is deprecated), pass 'use_fast_tsne=True'. Note, MulticoreTSNE is not actually faster anymore.
  warnings.warn(
/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:452: RuntimeWarning: overflow encountered in expm1
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (
/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:452: RuntimeWarning: overflow encountered in expm1
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (
/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:429: PerformanceWarning: DataFrame is highly fragmented.  This is usual

## 3. Process object with Harmony

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
adata = process_adata(adata, resolution=0.5, run_harmony=True, run_rank_genes=True)
adata.write('../../../data/rna/final-objects/final-bmmc-processed-harmony.h5ad')

2025-09-03 17:07:28,076 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2025-09-03 17:08:17,457 - harmonypy - INFO - sklearn.KMeans initialization complete.
2025-09-03 17:08:21,830 - harmonypy - INFO - Iteration 1 of 10
2025-09-03 17:15:14,434 - harmonypy - INFO - Iteration 2 of 10
2025-09-03 17:22:01,141 - harmonypy - INFO - Iteration 3 of 10
2025-09-03 17:28:49,055 - harmonypy - INFO - Iteration 4 of 10
2025-09-03 17:36:38,122 - harmonypy - INFO - Iteration 5 of 10
2025-09-03 17:44:34,682 - harmonypy - INFO - Converged after 5 iterations


## 4. Process L1 clusters with and without harmony

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
clusters = list(adata.obs['aifi_celltype_l1'].value_counts().index)
clusters.reverse()

for cluster in clusters:
    print(f'Processing {cluster}')
    subset = adata[adata.obs['aifi_celltype_l1'] == cluster]
    subset = process_adata(subset, resolution=1, run_harmony=True, run_rank_genes=True)
    subset.write(f'../../../data/rna/bmmc-celltypes/bmmc-{cluster}-processed-harmony.h5ad')

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
clusters = list(adata.obs['aifi_celltype_l1'].value_counts().index)
clusters.reverse()

for cluster in clusters:
    print(f'Processing {cluster}')
    subset = adata[adata.obs['aifi_celltype_l1'] == cluster]
    subset = process_adata(subset, resolution=1, run_harmony=False, run_rank_genes=True)
    subset.write(f'../../../data/rna/bmmc-celltypes/bmmc-{cluster}-processed.h5ad')